In [1]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

In [5]:
class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name
        self.http_args = {'headers': {'Accept': 'application/json'}}
        
    def _get_uniprot(self, accession):
        endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
        http_function = requests.get
        return http_function(endpoint, **self.http_args)

    def _get_ensembl(self, id):
        endpoint = f"https://rest.ensembl.org/lookup/id/{id}?expand=1"
        http_function = requests.get
        return http_function(endpoint, **self.http_args)

    def _uniprot_parse_response(self, resp: dict):
        if resp.status_code != 200:
            return {"error": f"HTTP error: {resp.status_code}"}
        resp = resp.json()
        output = {}
        acc = resp['primaryAccession']
        organism = resp['organism']['scientificName']
        geneInfo = resp['genes']
        sequenceInfo = resp['sequence']
        output[acc] = {'organism':organism, 'geneInfo':geneInfo, 'sequenceInfo':sequenceInfo, 'type':'protein'}
        return output

    def _ensembl_parse_response(self, resp):
        if resp.status_code != 200:
            return {"error": f"HTTP error: {resp.status_code}"}
        resp = resp.json()
        output = {}
        acc = resp['id']
        output[acc] = {
            'object_type': resp.get('object_type', ''),
            'assembly_name': resp.get('assembly_name', ''),
            'species': resp.get('species', ''),
            'db_type': resp.get('db_type', ''),
            'biotype': resp.get('biotype', ''),
            'display_name': resp.get('display_name', ''),
            'id': acc,
            'description': resp.get('description', 'No description available'),
            'canonical_transcript': resp.get('canonical_transcript', ''),
            'source': resp.get('source', '')
        }
        return output

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        # this function calls either _get_uniprot() or _get_ensembl()
        # an then _uniprot_parse_response() or _ensembl_parse_response()
        # outputs a dictionary with results (see example in test_data)
        output = {
            'description': seq_description,
            'sequence': seq_sequence,
            'database': 'Uniprot' if database == 'uniprot' else 'ENSEMBL',
        }
        
        if database == "uniprot":
            response = self._get_uniprot(id)
            parsed_data = self._uniprot_parse_response(response)
            
            if 'error' not in parsed_data:
                output['db_info'] = parsed_data
            else:
                output['db_info'] = parsed_data
                
        elif database == "ensembl":
            response = self._get_ensembl(id)
            output['db_info'] = self._ensembl_parse_response(response)
        else:
            output['db_info'] = {'error': f'Unknown database: {database}'}
        
        return output

    def seqkit_stats(self) -> dict:
        # this function calls seqkit via subprocess
        # if error arises, returns stderr and finishes the parser execution
        # if stats are collected, return the result (see example in test_data)
        result = subprocess.run(['seqkit', 'stats', '--all', self.filename], 
                              capture_output=True, text=True, check=True)
        
        lines = result.stdout.strip().split('\n')
        if len(lines) < 2:
            return {'error': 'No data returned from seqkit'}
        
        headers_raw = lines[0].split()
        data_raw = lines[1].split()
        
        headers = [h.strip('#').lower().strip() for h in headers_raw[1:]]
        data = data_raw[1:]
        
        stats_dict = dict(zip(headers, data))
        
        fasta_type = stats_dict.get('type', 'Unknown')
        num_seqs = int(stats_dict.get('num_seqs', 0))
        
        return {
            'fasta_seqkit_stat_info': stats_dict,
            'fasta_type': fasta_type,
            'fasta_num_seqs': num_seqs
        }
        
    def identify_database(self, description, seq_type):
        import re
        
        if seq_type == 'Protein':
            # FIXED regex - proper pipe escaping
            match = re.search(r'\b(?:sp|tr)\|([A-Z0-9]{6})\|', description)
            if match:
                return match.group(1), 'uniprot'
        elif seq_type == 'DNA':
            match = re.search(r'(ENS[A-Z0-9]+)', description)
            if match:
                return match.group(1), 'ensembl'
        
        return None, None
        
    def biopython_parser(self, seqkit_result) -> dict:
        # accepts seqkit_result dict
        # depending on fasta file type selects regular expression
        # parses fasta file via biopython
        # iterates over sequences and calls relevant database via _access_databases()
        # returns final result (info about each sequence from FASTA file + info from database)
        # Check if seqkit_stats returned an error
        fasta_type = seqkit_result.get('fasta_type', 'Unknown')
        output = {}
        warnings = []
        db_name = None
        
        for record in SeqIO.parse(self.filename, "fasta"):
            description = record.description
            sequence = str(record.seq)
            
            db_id, database = self.identify_database(description, fasta_type)
            
            if db_id and database:
                if db_name is None:
                    db_name = database  # This will be 'uniprot' or 'ensembl'
                    output['DB_name'] = db_name
                
                db_response = self._access_database(db_id, database, description, sequence)
                
                seq_key = f"file_info_{db_id}"
                output[seq_key] = {
                    'description': description,
                    'sequence': sequence
                }
                db_info_key = f"database_info_{db_id}"
                db_info_content = db_response['db_info']
                
                if isinstance(db_info_content, dict) and db_id in db_info_content:
                    output[db_info_key] = db_info_content[db_id]
                else:
                    output[db_info_key] = db_info_content
                
            else:
                warnings.append(f'No ID match for {record.id}')

        if warnings:
            output['WARNING'] = warnings
        return output
            
    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [20]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
stats
 # What is needed to get
 # {'fasta_seqkit_stat_info': {'format': 'FASTA',
 #  'type': 'Protein',
 #  'num_seqs': '2',
 #  'sum_len': '456',
 #  'min_len': '29',
 #  'avg_len': '228',
 #  'max_len': '427',
 #  'Q1': '29',
 #  'Q2': '228',
 #  'Q3': '427',
 #  'sum_gap': '0',
 #  'N50': '427',
 #  'N50_num': '1',
 #  'Q20(%)': '0',
 #  'Q30(%)': '0',
 #  'AvgQual': '0',
 #  'GC(%)': '0',
 #  'sum_n': '0'},
 # 'fasta_type': 'Protein',
 # 'fasta_num_seqs': 2}

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228',
  'max_len': '427',
  'q1': '29',
  'q2': '228',
  'q3': '427',
  'sum_gap': '0',
  'n50': '427',
  'n50_num': '1',
  'q20(%)': '0',
  'q30(%)': '0',
  'avgqual': '0',
  'gc(%)': '0',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [21]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)
# What is needed to get
# DB_name
# 	uniprot
# file_info_P11473
# 	description
# 		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
# 	sequence
# 		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
# database_info_P11473
# 	organism
# 		Homo sapiens
# 	geneInfo
# 		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
# 	sequenceInfo
# 		value
# 			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
# 		length
# 			427
# 		molWeight
# 			48289
# 		crc64
# 			F95F300D042C4CB7
# 		md5
# 			0D963ACD4A34674368324EE026023597
# 	type
# 		protein
# WARNING
# 	{'No ID match found.'}

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

In [6]:
parser = MyFastaParser('ensembl_download_1.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'DNA',
  'num_seqs': '6',
  'sum_len': '86',
  'min_len': '9',
  'avg_len': '14.3',
  'max_len': '23',
  'q1': '10',
  'q2': '13.5',
  'q3': '17',
  'sum_gap': '0',
  'n50': '16',
  'n50_num': '3',
  'q20(%)': '0',
  'q30(%)': '0',
  'avgqual': '0',
  'gc(%)': '45.35',
  'sum_n': '0'},
 'fasta_type': 'DNA',
 'fasta_num_seqs': 6}

In [7]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	object_type
		Transcript
	assembly_name
		GRCm39
	species
		mus_musculus
	db_type
		core
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	description
		No description available
	canonical_transcript
		
	source
		havana
file_info_ENSMUST00000177564
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
database_info_ENSMUST00000177564
	object_type
		Transcript
	assembly_name
		GRC

In [9]:
parser = MyFastaParser('ensembl_download_2.fasta')
stats = parser.seqkit_stats()
stats

{'error': 'No data returned from seqkit'}

In [10]:
parser = MyFastaParser('uniprot_download.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '7',
  'sum_len': '3,861',
  'min_len': '180',
  'avg_len': '551.6',
  'max_len': '1,382',
  'q1': '429',
  'q2': '441',
  'q3': '500',
  'sum_gap': '0',
  'n50': '468',
  'n50_num': '3',
  'q20(%)': '0',
  'q30(%)': '0',
  'avgqual': '0',
  'gc(%)': '0',
  'sum_n': '0'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 7}

In [11]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLL